In [8]:
import pandas as pd
import requests
import json

# Weather data source: NOAA National Centers for Environmental Information (NCEI).

TOKEN = "fmMZOwfoaPDSXZNdwClrfpATwtcoFXOs"

headers = {
    "token": TOKEN
}

In [ ]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:US1WIMW0008",
    "startdate": "2025-10-01",
    "enddate": "2026-06-30",
    "datatypeid": ["TMAX", "TMIN", "PRCP", "SNOW", "SNWD"],
    "units": "standard",
    "limit": 1000
}



response = requests.get(url, headers=headers, params=params)
print(response.json())

{}


In [16]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"

params = {
    "datasetid": "LCD",
    "stationid": "WBAN:04897"
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

parsed_data = response.json()

print(json.dumps(parsed_data, indent=2, sort_keys=True))

KeyboardInterrupt: 

In [17]:
# 1. Verify the station exists in LCD
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

params = {
    "datasetid": "LCD",
    "stationid": "WBAN:04897"
}

response = requests.get(url, headers=headers, params=params)
print(response.status_code)
print(response.text)

200
{"metadata":{"resultset":{"offset":1,"count":2577,"limit":25}},"results":[{"elevation":55.5,"mindate":"2012-09-25","maxdate":"2012-11-23","latitude":34.09972,"name":"DEXTER B FLORENCE MEMORIAL FIELD AIRPORT, AR US","datacoverage":1,"id":"WBAN:00100","elevationUnit":"METERS","longitude":-93.06583},{"elevation":637,"mindate":"2020-09-08","maxdate":"2020-09-27","latitude":43.067,"name":"MANAS INTERNATIONAL AIRPORT, KG","datacoverage":0.26,"id":"WBAN:00101","elevationUnit":"METERS","longitude":74.483},{"elevation":51.2,"mindate":"2008-08-22","maxdate":"2025-08-25","latitude":66.983,"name":"BOB BARKER MEMORIAL AIRPORT, AK US","datacoverage":1,"id":"WBAN:00102","elevationUnit":"METERS","longitude":-160.433},{"elevation":7,"mindate":"2009-01-22","maxdate":"2025-08-25","latitude":65.617,"name":"WALES AIRPORT, AK US","datacoverage":1,"id":"WBAN:00103","elevationUnit":"METERS","longitude":-168.1},{"elevation":126.2,"mindate":"2006-10-24","maxdate":"2025-08-25","latitude":63.017,"name":"NIKOL

In [20]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

params = {
    "datasetid": "LCD",
    "stationid": "WBAN:04897",
    "startdate": "2025-10-01",
    "enddate": "2025-10-31",
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)
print(response.status_code)
print(response.text)

500
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><statusCode>500</statusCode><userMessage>An error occured while servicing your request.</userMessage><developerMessage>An error occured while servicing your request.</developerMessage></response>


In [21]:
from io import StringIO
import pandas as pd
import requests

url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "local-climatological-data",
    "stations": "WBAN:04897",
    "startDate": "2025-10-01",
    "endDate": "2025-10-31",
    "format": "csv"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.text[:500])

df = pd.read_csv(StringIO(response.text))

print(df.head())
print(df.columns)

200
"STATION","DATE","REPORT_TYPE","SOURCE","AWND","BackupDirection","BackupDistance","BackupDistanceUnit","BackupElements","BackupElevation","BackupElevationUnit","BackupEquipment","BackupLatitude","BackupLongitude","BackupName","CDSD","CLDD","DSNW","DYHF","DYTS","DailyAverageDewPointTemperature","DailyAverageDryBulbTemperature","DailyAverageRelativeHumidity","DailyAverageSeaLevelPressure","DailyAverageStationPressure","DailyAverageWetBulbTemperature","DailyAverageWindSpeed","DailyCoolingDegreeDays
Empty DataFrame
Columns: [STATION, DATE, REPORT_TYPE, SOURCE, AWND, BackupDirection, BackupDistance, BackupDistanceUnit, BackupElements, BackupElevation, BackupElevationUnit, BackupEquipment, BackupLatitude, BackupLongitude, BackupName, CDSD, CLDD, DSNW, DYHF, DYTS, DailyAverageDewPointTemperature, DailyAverageDryBulbTemperature, DailyAverageRelativeHumidity, DailyAverageSeaLevelPressure, DailyAverageStationPressure, DailyAverageWetBulbTemperature, DailyAverageWindSpeed, DailyCoolingDegreeD

In [13]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

lat = 43.07044610327365
lon = -88.12440893206329

# Bigger search box around the coordinates
extent = f"{lat - 0.5},{lon - 0.5},{lat + 0.5},{lon + 0.5}"

params = {
    "datasetid": "LCD",
    "extent": extent,
    "limit": 25
}

response = requests.get(url, headers=headers, params=params)

response.raise_for_status()

stations = response.json()

if "results" in stations:
    for station in stations["results"]:
        print(station["id"], "-", station["name"])
else:
    print(stations)

WBAN:04845 - KENOSHA REGIONAL AIRPORT, WI US
WBAN:04866 - BURLINGTON MUNICIPAL AIRPORT, WI US
WBAN:04875 - WEST BEND MUNICIPAL AIRPORT, WI US
WBAN:04897 - WAUKESHA CO AIRPORT, WI US
WBAN:14839 - MILWAUKEE MITCHELL AIRPORT, WI US
WBAN:94818 - RACINE BATTEN AIRPORT, WI US
WBAN:94869 - MILWAUKEE TIMMERMAN AIRPORT, WI US


In [15]:
params = {
    "datasetid": "LCD",
    "locationid": "FIPS:55",   # Wisconsin
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)

stations = response.json()

for station in stations.get("results", []):
    if "WAUKESHA" in station["name"].upper():
        print(station["id"], "-", station["name"])